# 02 — Hybrid retrieval and RRF

Reproduce the article's RRF top-3 (`d3`, `d2`, `d1`), then compare dense / sparse / hybrid on
the live scifact slice.


In [ ]:
from rag_evals.retrieval.hybrid_rrf import reciprocal_rank_fusion

dense  = ["d3", "d7", "d1", "d4", "d2", "d9", "d10"]
sparse = ["d2", "d3", "d8", "d1", "d11", "d4", "d6"]
for doc, score in reciprocal_rank_fusion([dense, sparse], k=60)[:5]:
    print(f"  {doc}  score={score:.5f}")


In [ ]:
import json
from rag_evals.config import settings
from rag_evals.evaluation.retrieval import evaluate_runs
from rag_evals.retrieval.dense import DenseRetriever
from rag_evals.retrieval.sparse import SparseRetriever
from rag_evals.retrieval.hybrid_rrf import HybridRetriever

rows = [json.loads(l) for l in (settings.golden_dir / "retrieval.jsonl").open()][:50]

dense = DenseRetriever()
sparse = SparseRetriever()
hybrid = HybridRetriever(dense, sparse, k=60)

results = {"dense": {}, "sparse": {}, "hybrid": {}}
gold = {}
for r in rows:
    qid, q = r["qid"], r["query"]
    results["dense"][qid]  = [h.doc_id for h in dense(q, limit=10)]
    results["sparse"][qid] = [h.doc_id for h in sparse(q, limit=10)]
    results["hybrid"][qid] = [h.doc_id for h in hybrid(q, limit=10)]
    gold[qid] = r["gold_doc_ids"]

for name, runs in results.items():
    m = evaluate_runs(runs, gold, k=10)
    print(f"{name:>7}: Recall@10={m.recall_at_k:.3f} MRR={m.mrr:.3f} nDCG@10={m.ndcg_at_k:.3f}")


On the article's claim: "hybrid Recall@10 ≥ max(dense, sparse)" — true on most corpora;
verify on yours before trusting it.
